# Trích xuất feature ONE-PEACE cho UniAV (YouCookII)

**Chỉ cần sửa ô số 2 (ĐƯỜNG DẪN).** Các ô sau dùng lại biến ở đó.

Code (clone từ GitHub) và 2 file checkpoint (trên Google Drive) có thể nằm ở hai nơi khác nhau: đường dẫn checkpoint khai báo riêng bằng `VIDEO_CKPT` và `AUDIO_CKPT`.

Thứ tự: 1 → 2 → 3, sau đó chạy phần **A (visual)** hoặc **B (audio)**. Mỗi lần Colab khởi động lại phải chạy lại 1 → 2 → 3 và bước cài đặt của phần tương ứng.

Runtime → Change runtime type → **GPU (khuyên dùng A100)**.

In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 2. ĐƯỜNG DẪN — SỬA Ở ĐÂY
# Thư mục code One_Peace (bản clone từ GitHub), ví dụ '/content/drive/MyDrive/KL/One_Peace' hoặc '/content/One_Peace'
ONE_PEACE_DIR = "/content/drive/MyDrive/KL/One_Peace"

# 2 file checkpoint trên Google Drive (không nằm trong repo code)
VIDEO_CKPT = "/content/drive/MyDrive/KL/checkpoints/onepeace_video_k400.pth"  # checkpoint visual (6.6 GB)
AUDIO_CKPT = "/content/drive/MyDrive/KL/checkpoints/one-peace-audio.pt"  # checkpoint audio (5.7 GB)

VIDEO_DIR = "/content/drive/MyDrive/KL/YouCookII/videos"  # thư mục chứa <video_id>.mp4 (còn âm thanh — phần A lấy hình, phần B lấy tiếng)
FEAT_DIR = "/content/drive/MyDrive/KL/feats/youcookii"  # nơi ghi *_one_peace_video_finetune.npy và *_one_peace_audio.npy
ANNO_JSON = f"{ONE_PEACE_DIR}/annotations/youcookii_all.json"  # danh sách video cần xử lý (có sẵn trong repo)

# Tham số (giữ nguyên để sát bài báo)
STRIDE = 8  # bước visual theo frame @16fps: 8 = 0.5 s (như ActivityNet), 4 = 0.25 s
STRIDE_SEC = STRIDE / 16  # bước audio tương ứng, KHÔNG sửa riêng
BATCH_VIDEO = 16  # số clip / lần forward: A100 dùng 16, T4 dùng 8; giảm nếu hết VRAM
DTYPE_VIDEO = "fp16"  # fp16: cos >= 0.9999 so với fp32, nhanh ~3.4 lần; bf16 kém chính xác hơn (cos ~0.998); fp32 rất chậm
COMPILE_VIDEO = True  # torch.compile: nhanh hơn, tốn ~1 phút biên dịch lúc đầu
BATCH_AUDIO = 64  # số cửa sổ 1 s / lần forward

# Chia việc cho nhiều phiên Colab: phiên thứ k đặt SHARD_ID = k (0 .. NUM_SHARDS-1)
NUM_SHARDS = 1
SHARD_ID = 0

# Bản copy checkpoint trên ổ local của Colab (nạp nhanh hơn đọc từ Drive) — không cần sửa
VIDEO_CKPT_LOCAL = "/content/ckpt/onepeace_video_k400.pth"
AUDIO_CKPT_LOCAL = "/content/ckpt/one-peace-audio.pt"

In [ ]:
# 3. Kiểm tra đường dẫn + GPU
import glob
import os

for p in [ONE_PEACE_DIR, VIDEO_CKPT, AUDIO_CKPT, VIDEO_DIR, ANNO_JSON]:
    print("OK   " if os.path.exists(p) else "THIẾU", p)
print("số video mp4:", len(glob.glob(f"{VIDEO_DIR}/*.mp4")))
os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs("/content/ckpt", exist_ok=True)
%cd {ONE_PEACE_DIR}
!nvidia-smi --query-gpu=name,memory.total --format=csv

## A. Visual (Python mặc định của Colab)

In [ ]:
# A1. Cài đặt + copy checkpoint visual từ Drive ra ổ local của Colab
!pip install -q einops
!cp -n "{VIDEO_CKPT}" "{VIDEO_CKPT_LOCAL}"
!ls -la "{VIDEO_CKPT_LOCAL}"

In [ ]:
# A2. Kiểm tra nhanh: clip giây 92 phải ra 'making a sandwich'; in thêm sai khác fp16/bf16 vs fp32
!python sanity_check_video.py --checkpoint "{VIDEO_CKPT_LOCAL}"     --video "{VIDEO_DIR}/GLd3aX16zBg.mp4" --start_sec 92 --compare_fp16

In [ ]:
# A3. Đo tốc độ trên 400 clip (ghi ra /content/bench, không đụng FEAT_DIR).
# Với COMPILE_VIDEO = True, vài batch đầu gồm cả thời gian biên dịch -> xem tốc độ ở các dòng tiến độ sau.
COMPILE_FLAG = "--compile" if COMPILE_VIDEO else ""
!python extract_video_features.py --checkpoint "{VIDEO_CKPT_LOCAL}"     --video_dir "{VIDEO_DIR}" --output_dir /content/bench --ids_from "{ANNO_JSON}"     --stride {STRIDE} --batch_size {BATCH_VIDEO} --dtype {DTYPE_VIDEO} {COMPILE_FLAG}     --limit 1 --max_clips 400 --overwrite --log_every 20

In [ ]:
# A4. Chạy thật. Bị ngắt thì chạy lại ô này: video đã có file .npy sẽ được bỏ qua
COMPILE_FLAG = "--compile" if COMPILE_VIDEO else ""
!python extract_video_features.py --checkpoint "{VIDEO_CKPT_LOCAL}"     --video_dir "{VIDEO_DIR}" --output_dir "{FEAT_DIR}" --ids_from "{ANNO_JSON}"     --stride {STRIDE} --batch_size {BATCH_VIDEO} --dtype {DTYPE_VIDEO} {COMPILE_FLAG}     --num_shards {NUM_SHARDS} --shard_id {SHARD_ID}

## B. Audio (cần môi trường Python 3.10 riêng cho fairseq cũ của ONE-PEACE)

Audio được lấy thẳng từ track âm thanh của `VIDEO_DIR/<video_id>.mp4`, giải mã và resample giống hệt `librosa.load(sr=16000)` trong code gốc ONE-PEACE.

In [ ]:
# B1. Cài đặt (~3-5 phút) + copy checkpoint audio từ Drive ra ổ local của Colab
!git clone -q --depth 1 https://github.com/OFA-Sys/ONE-PEACE /content/ONE-PEACE || true
!pip install -q uv && uv venv -q --seed --python 3.10 /content/op310
!/content/op310/bin/python -m pip install -q "pip==24.0"
# torch và torchvision phải cài cùng lệnh, nếu không timm sẽ kéo về bản torch mới nhất (bản CPU)
!/content/op310/bin/python -m pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
# setuptools >= 81 bỏ pkg_resources mà librosa 0.10.0 vẫn import; resampy cần cho --res_type kaiser_best
!/content/op310/bin/python -m pip install -q "setuptools<81" "numpy<2" hydra-core==1.0.7 omegaconf==2.0.6     antlr4-python3-runtime==4.8 bitarray sacrebleu tabulate regex timm==0.6.11 iopath tensorboardX pydub     librosa==0.10.0 resampy soundfile soxr einops opencv-python-headless scipy tqdm pillow imageio-ffmpeg
!cp -n "{AUDIO_CKPT}" "{AUDIO_CKPT_LOCAL}"
!/content/op310/bin/python -c "import torch, librosa; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!ls -la "{AUDIO_CKPT_LOCAL}"

In [ ]:
# B2. Chạy thật. Bị ngắt thì chạy lại ô này
# Mặc định: --dtype auto = fp16 trên GPU, --res_type kaiser_best (khớp nhất với feature của tác giả, xem README mục 9)
!/content/op310/bin/python extract_audio_features.py --onepeace_repo /content/ONE-PEACE     --checkpoint "{AUDIO_CKPT_LOCAL}"     --video_dir "{VIDEO_DIR}" --output_dir "{FEAT_DIR}" --ids_from "{ANNO_JSON}"     --stride_sec {STRIDE_SEC} --batch_size {BATCH_AUDIO} --num_shards {NUM_SHARDS} --shard_id {SHARD_ID}

## C. Kiểm tra kết quả (sau khi xong cả A và B)

In [ ]:
!python check_features.py --anno "{ANNO_JSON}" --feat_dir "{FEAT_DIR}" --stride {STRIDE}